# Train MuJoCo Playground's SO-101 PickCube for mjswan

MuJoCo Playground 1.0.1's SO-101 PickCube, trained with Playground's own environment and
PPO settings, then exported as the ONNX policy an mjswan scene runs. `TASK` picks the
variant: `"state"` (`SO101PickCube`: joints, gripper and cube positions; under an hour on
a T4) or `"vision"` (`SO101PickCubeVision`, the task Playground's sim-to-real demo deploys:
the wrist camera's image instead of the cube's position; hours, over several sessions).

1. **Runtime → Change runtime type → T4 GPU**.
2. Add a Hugging Face **write** token as the Colab secret `HF_TOKEN` (the key icon on the
   left), with notebook access on.
3. Set `TASK` and `HF_REPO` below and run the cells in order. Every checkpoint is kept in
   that repository, so when a session ends, running the cells again resumes from the last
   one.
4. The result is in the same repository: `policy.onnx`, `contract.json` and `reference.npz`
   (see [`so101_pick_cube.py`](https://github.com/ttktjmt/mjswan/blob/main/examples/train/so101_pick_cube.py)).

In [ ]:
!nvidia-smi

In [ ]:
import os

from google.colab import userdata

# "state" or "vision".
TASK = "state"
# A Hugging Face model repository, "user/name", created public if it does not exist. One
# per task: a run resumes from whatever checkpoint the repository holds.
HF_REPO = ""
# None: Playground's own, 1000 for state and 3000 for vision.
ITERATIONS = None
# The mjswan branch the training script is fetched from.
BRANCH = "main"

if not HF_REPO:
    raise ValueError("Set HF_REPO: it keeps what an ended session loses.")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
OUT = f"/content/so101_pick_cube_{TASK}"
RUN = f"--task {TASK} --out {OUT} --hf-repo {HF_REPO}"
if ITERATIONS:
    RUN += f" --iterations {ITERATIONS}"

In [ ]:
!wget -q -O so101_pick_cube.py https://raw.githubusercontent.com/ttktjmt/mjswan/{BRANCH}/examples/train/so101_pick_cube.py
!python so101_pick_cube.py setup

Training curves while it runs (optional; this session's only).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$OUT/checkpoints"

Train. If the session ends, run the cells above again and then this one: it resumes from
the newest checkpoint in `HF_REPO`.

In [ ]:
!python so101_pick_cube.py train {RUN}

In [ ]:
!python so101_pick_cube.py export {RUN}
!python so101_pick_cube.py upload {RUN}